Dataset: day-03-evidence/source-extract/insurance policy data.xlsx
Business scenario: trusted policyholder / claimant master for insurance operation.
Goal: create quality checks, contract, classification, lineage and golden-record evidence.

In [2]:
from pathlib import Path 
import pandas as pd 

primary = Path('/home/labuser/Desktop/Persistent_Folder/Workspace/day-03-evidence/source-extract/insurance policy data.xlsx')

if primary.exists():
    dataset_path = primary
    df = pd.read_excel(primary)
    dataset_used = 'primary insurance policy data'
else:
    raise FileNotFoundError('Primary dataset not found.')

print('dataste_used: ', dataset_used)
print('dataste_path: ', dataset_path)
print('rows: ', len(df))
print('columns: ', list(df.columns))
df.head(5)

dataste_used:  primary insurance policy data
dataste_path:  /home/labuser/Desktop/Persistent_Folder/Workspace/day-03-evidence/source-extract/insurance policy data.xlsx
rows:  1000
columns:  ['Name', 'Date', 'Shift Timing', 'Team Name', 'Activity', 'Job Number', 'policy category', 'Job Type', 'Fund', 'Sub Fund', 'Final Action', 'Average handling time (minutes)', 'country']


,Name,Date,Shift Timing,Team Name,Activity,Job Number,policy category,Job Type,Fund,Sub Fund,Final Action,Average handling time (minutes),country
0,Monika,2011-12-15,09:00 - 17:00,Policy Ops,Escalation,1000825,corporate,Address Change,SA003,REF001,Routed,25,australia
1,Nikhil,2025-11-16,09:00 - 17:00,Member Services,Audit,1000205,corporate,Update Member Insurance,SA002,RT001,Processed,22,singapore
2,Tanya,2017-11-05,09:00 - 17:00,Claims,Audit,1000962,student,Inquiry Insurance Quote,SA003,HLT004,Processed,17,usa
3,Arjun,2003-01-03,09:00 - 17:00,Quality,Audit,1000785,group,Policy Cancellation,SA001,ING002,Routed,23,india
4,Ira,2025-10-18,09:00 - 17:00,Claims,Audit,1000616,family,Insurance Life Event,SA021,HLT004,Rejected,29,australia


### 3.1 Dataset Load proof
- Dataset Used: captured
- Row Count: 1000
- Columns List: visible
- First visible output : yes
- limitation: this proves the file is loaded; it does not yet prove quality, governance or MDM

In [3]:
import pandas as pd 

columns = list(df.columns)
lower_map = { str(c).lower():  c for c in columns }

id_candidates =  [c for c in columns if any(token in str(c).lower() for token in ['policy', 'customer', 'claim', 'id'])]
date_candidates =  [c for c in columns if any(token in str(c).lower() for token in  ['date', 'dob', 'start', 'end'])]
status_candidates =  [c for c in columns if any(token in str(c).lower() for token in  ['status', 'type', 'category'])]


primary_key_guess = id_candidates[0] if id_candidates else columns[0]
date_guess = date_candidates[0] if date_candidates else None
status_guess = status_candidates[0] if status_candidates else None 

quality_rows = []

quality_rows.append({
    'dimension': 'completeness',
    'rule': f'{primary_key_guess} should not be null',
    'failing_rows': int(df[primary_key_guess].isna().sum()),
    'evidence': 'null count on guessed key column',
    'needs_business_confirmation': True,
})

quality_rows.append({
    'dimension': 'validity',
    'rule': f'{primary_key_guess} should not be duplicated',
    'failing_rows': int(df.duplicated(subset=[primary_key_guess]).sum()),
    'evidence': 'duplicate count on guessed key column',
    'needs_business_confirmation': True,
})

if date_guess:
    parsed_dates = pd.to_datetime(df[date_guess], errors='coerce')
    quality_rows.append({
        'dimension': 'timeliness',
        'rule': f'{date_guess} should parse as a date',
        'failing_rows': int(parsed_dates.isna().sum()),
        'evidence': 'date parse failure count',
        'needs_business_confirmation': True,
    })

if status_guess:
    top_values = df[status_guess].astype(str).value_counts(dropna=False).head(10).to_dict()
    quality_rows.append({
        'dimension': 'distribution',
        'rule': f'{status_guess} distribution should be reviewed for unexpected values',
        'failing_rows': 'review required',
        'evidence': top_values,
        'needs_business_confirmation': True,
    })

quality_report = pd.DataFrame(quality_rows)

quality_report



,dimension,rule,failing_rows,evidence,needs_business_confirmation
0,completeness,policy category should not be null,0,null count on guessed key column,True
1,validity,policy category should not be duplicated,994,duplicate count on guessed key column,True
2,timeliness,Date should parse as a date,0,date parse failure count,True
3,distribution,policy category distribution should be reviewe...,review required,"{'corporate': 179, 'student': 178, 'individual...",True


### Data Contract Draft

1. Producer
2. Consumer
3. Grain
4. Required fields
5. Freshness SLA
6. Quality SLA
7. PII handling
8. Change Handling
9. Owner
10. Human approval point

In [4]:
observability_snapshot = pd.DataFrame([
    {
        'monitor': 'freshness',
        'signal': 'date column parse / latest date',
        'status': 'review_required'
    },
    {
        'monitor': 'volumne',
        'signal': f'row_count={len(df)}',
        'status': 'captured'
    },
    {
        'monitor': 'schema',
        'signal': f'column_count={len(df.columns)}',
        'status': 'captured'
    },
    {
        'monitor': 'distribution',
        'signal': 'top values reviewed for candidate status/category column',
        'status': 'review_required'
    },
])

observability_snapshot

,monitor,signal,status
0,freshness,date column parse / latest date,review_required
1,volumne,row_count=1000,captured
2,schema,column_count=13,captured
3,distribution,top values reviewed for candidate status/categ...,review_required


In [2]:
import pandas as pd 
from difflib import SequenceMatcher

sample_entities = pd.DataFrame([
    {
        'source_system': 'policy_admin',
        'source_record_id': 'PA-1001',
        'customer_name': 'A Sharma',
        'phone': '9999992345',
        'email': 'a.sharma@example.com',
        'policy_id': 'POL-101',
        'updated_at': '2026-07-20'
    },
    {
        'source_system': 'claims',
        'source_record_id': 'CL-7788',
        'customer_name': 'Abhishek Sharma',
        'phone': '9999992345',
        'email': 'abhishek.sharma@example.com',
        'policy_id': 'POL-101',
        'updated_at': '2026-07-25'
    },
    {
        'source_system': 'claims',
        'source_record_id': 'CL-8899',
        'customer_name': 'R Mehta',
        'phone': '8888881111',
        'email': 'r.mehta@example.com',
        'policy_id': 'POL-222',
        'updated_at': '2026-07-25'
    },
])

def similiarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

left = sample_entities.iloc[0]
right =  sample_entities.iloc[1]
name_score = similiarity(left['customer_name'], right['customer_name'])
phone_score = 1.0 if left['phone'] == right['phone'] else 0.0
policy_score = 1.0 if left['policy_id'] == right['policy_id'] else 0.0
match_score = round((name_score * 0.4) + (phone_score * 0.3) + (policy_score * 0.3) , 3)

candidate = {
    'candidate_pair': f"{left['source_record_id']} + { right['source_record_id'] }",
    'match_score': match_score,
    'proposed_golden_name': right['customer_name'] if len(right['customer_name']) > len(left['customer_name']) else left['customer_name'],
    'proposed_policy_id': left['policy_id'],
    'proposed_phone': left['phone'],
    'reason': 'same phone and policy id; similiar name',
    'risk': 'wrong merge could attach claim history to wrong person',
    'steward_decision_required': True,
}

pd.DataFrame([candidate])

,candidate_pair,match_score,proposed_golden_name,proposed_policy_id,proposed_phone,reason,risk,steward_decision_required
0,PA-1001 + CL-7788,0.878,Abhishek Sharma,POL-101,9999992345,same phone and policy id; similiar name,wrong merge could attach claim history to wron...,True
